In [29]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import math 

import sys 
import os

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)


import tarfile
import urllib

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats/cookie_cats.csv


# Exploratory Data Analysis

We first check that the experiment was randomized correctly (group sizes) and get a feel for how players engage with the game before testing anything formally.

In [30]:
df = pd.read_csv('/kaggle/input/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats/cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [ ]:
########### visualizations ###########################

# Consistent colors for the two experiment groups, used across every chart below
GROUP_COLORS = {"gate_30": "#2a78d6", "gate_40": "#1baf7a"}
GROUP_LABELS = {"gate_30": "gate_30 (Control)", "gate_40": "gate_40 (Test)"}

# 1) Imbalance of control and test group

fig, ax = plt.subplots(figsize=(7,5))
version_counts = df["version"].value_counts()
ax.pie(version_counts,
       labels=[GROUP_LABELS[v] for v in version_counts.index],
       colors=[GROUP_COLORS[v] for v in version_counts.index],
       autopct="%1.1f%%",
       wedgeprops={"edgecolor": "white", "linewidth": 1})
ax.set_title("A/B group split — near-even 50/50 randomization")
plt.show()
plt.close()

# 2) Gamerounds distribution

fig, axs = plt.subplots(1, 2, figsize=(12,5))
axs = axs.flatten()

axs[0].hist(df["sum_gamerounds"], bins=100, color="#2a78d6")
axs[0].set_xlabel("Game rounds (14-day sum)")
axs[0].set_xscale('linear')
axs[0].set_ylabel("Players (log scale)")
axs[0].set_yscale('log')
axs[0].set_title("Gamerounds distribution — all players")

## Gamerounds distribution per group (clipped so the long tail doesn't hide the shape)

sns.kdeplot(
    data=df[df["sum_gamerounds"] < 1000],
    x="sum_gamerounds",
    hue="version",
    hue_order=["gate_30", "gate_40"],
    palette=GROUP_COLORS,
    ax=axs[1],
    fill=True,
    alpha=0.3
)
axs[1].set_xlabel("Game rounds")
axs[1].set_ylabel("Density")
axs[1].set_title("Gamerounds by group (clipped at 1000 rounds)")
legend = axs[1].get_legend()
legend.set_title("Group")
for text, version in zip(legend.get_texts(), ["gate_30", "gate_40"]):
    text.set_text(GROUP_LABELS[version])

plt.tight_layout()
plt.show()
plt.close()

## outliers detection
Q1 = df["sum_gamerounds"].quantile(0.25)
Q3 = df["sum_gamerounds"].quantile(0.75)
IQR = Q3 - Q1
### get the gamerounds in the outlier region
outliers = df[df["sum_gamerounds"] > Q3 + 3 * IQR]
print(f"Outliers (> Q3 + 3xIQR): {len(outliers)} players ({len(outliers)/len(df)*100:.2f}%) — kept in the data, excluded only from this plot for readability")
### gamerounds per group without the extreme outliers, so the boxes are still readable
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df[df["sum_gamerounds"] < Q3 + 3*IQR],
            x="version", y="sum_gamerounds", order=["gate_30", "gate_40"],
            hue="version", hue_order=["gate_30", "gate_40"], palette=GROUP_COLORS, legend=False,
            ax=ax)
ax.set_xticks([0, 1])
ax.set_xticklabels([GROUP_LABELS[v] for v in ["gate_30", "gate_40"]])
ax.set_xlabel("Group")
ax.set_ylabel("Game rounds")
ax.set_title("Gamerounds per group (mild outliers removed for scale)")
plt.show()
plt.close()


# 3) retention rates distributions

fig, axs = plt.subplots(1, 2, figsize=(8,5))
axs = axs.flatten()

for ax, col, title in zip(axs, ["retention_1", "retention_7"], ["Day-1 retention", "Day-7 retention"]):
    sns.countplot(data=df, x=col, ax=ax, color="#2a78d6", order=[False, True])
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Not retained", "Retained"])
    ax.set_xlabel("")
    ax.set_ylabel("Players")
    ax.set_title(title)
    total = len(df)
    for bar in ax.patches:
        pct = bar.get_height() / total * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f"{pct:.1f}%", ha="center", va="bottom", fontsize=10)

plt.suptitle("Overall retention (both groups combined)", y=1.02)
plt.tight_layout()
plt.show()
plt.close()

# retention rate per group

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(axs,
                           ["retention_1", "retention_7"],
                           ["Day-1 retention", "Day-7 retention"]):
    rates = df.groupby("version")[col].mean().reindex(["gate_30", "gate_40"]).reset_index()
    sns.barplot(data=rates, x="version", y=col, order=["gate_30", "gate_40"],
                hue="version", hue_order=["gate_30", "gate_40"], palette=GROUP_COLORS, legend=False,
                ax=ax)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([GROUP_LABELS[v] for v in ["gate_30", "gate_40"]])
    ax.set_ylim(0, 0.6)
    ax.set_xlabel("")
    ax.set_ylabel("Retention rate")
    ax.set_title(title)
    for bar, val in zip(ax.patches, rates[col]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontsize=11)
plt.suptitle("Retention rate by group — at face value, the groups look similar", y=1.02)
plt.tight_layout()
plt.show()
plt.close()

# retention rate per group, broken down by engagement depth (rounds played)

df["grouped_gamerounds"] = pd.cut(df["sum_gamerounds"],
                              bins=[0, 5, 20, 50, 100, 500, 50000],
                              labels=["0-5","6-20","21-50","51-100","101-500","500+"])

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, title in zip(axs, ["retention_1", "retention_7"],
                               ["Day-1 retention", "Day-7 retention"]):
    ct = df.groupby(["grouped_gamerounds", "version"], observed=True)[col].mean().unstack()[["gate_30", "gate_40"]]
    ct.plot(kind="bar", ax=ax, color=[GROUP_COLORS["gate_30"], GROUP_COLORS["gate_40"]])
    ax.set_title(f"{title} by rounds played")
    ax.set_xlabel("Gamerounds bucket")
    ax.set_ylabel("Retention rate")
    ax.tick_params(axis='x', rotation=30)
    ax.legend([GROUP_LABELS["gate_30"], GROUP_LABELS["gate_40"]], title="Group")
plt.suptitle("Retention increases with engagement depth in both groups", y=1.02)
plt.tight_layout()
plt.show()

## What do we observe from our EDA?

* The experiment is well-randomized: gate_30 (50.4%) and gate_40 (49.6%) are nearly equal in size, confirming no assignment bias.
* The distribution of gamerounds is heavily right-skewed. The vast majority of player activity is concentrated below 200 rounds, with a long tail of high-engagement outliers. The median (not the mean) is the appropriate central tendency measure.
* Approximately 6% of players show extreme round counts beyond Q3 + 3×IQR. These were retained as legitimate user behavior; robust statistics were used to minimize their influence on the plots above.
* Both groups show near-identical gamerounds distributions in shape, median, and spread, suggesting the gate position did not meaningfully affect overall engagement volume during the 14-day window.
* Day-7 retention (~18%) is significantly lower than Day-1 retention (~44%), indicating substantial churn between the first and seventh day — a normal pattern in mobile gaming.
* At face value, both groups show similar retention rates on Day-1 (~44%) and Day-7 (~18%). However, **gate_30 shows a marginally higher Day-7 retention, which warrants formal statistical testing to determine if the difference is significant.**
* Retention rate increases monotonically with engagement depth across both groups — **players who completed more rounds were substantially more likely to return on Day-1 and Day-7.**

# Hypothesis framing

## Null and alternative hypothesis

We compare the two gate positions on Day-7 retention:

* $H_{0}$: $P(\text{retain}_7 \mid \text{gate\_30}) = P(\text{retain}_7 \mid \text{gate\_40})$ — the gate position has no effect on Day-7 retention.
* $H_{1}$: $P(\text{retain}_7 \mid \text{gate\_30}) \neq P(\text{retain}_7 \mid \text{gate\_40})$ — the gate position affects Day-7 retention.

Primary metric: retention rate on Day 7. A two-tailed test is used because we have no prior reason to expect the effect in one specific direction.

In [32]:
# Proportions per group
gate30 = df[df["version"] == "gate_30"]
gate40 = df[df["version"] == "gate_40"]

# Sample sizes for each retention metric
n30, n40 = len(gate30), len(gate40)

# number of success for each group
r7_30, r7_40 = gate30["retention_7"].sum(), gate40["retention_7"].sum()

# Observed proportions (effect size reference)

print(f"Day-7: gate_30={r7_30/n30:.4f}, gate_40={r7_40/n40:.4f}")

Day-7: gate_30=0.1902, gate_40=0.1820


# Statistical test selection

Day-7 retention is a binary outcome (retained / not retained) measured on two independent, randomly assigned groups with a large sample size in each — the standard setting for a **two-proportion z-test**.

In [33]:
from statsmodels.stats.proportion import proportions_ztest

# two-tailed z-test 
z_stat, p_value = proportions_ztest(
    count = [r7_30, r7_40], # the number of successes in nobs trials.
    nobs = [n30, n40], # the number of trials or observations
    alternative = 'two-sided'
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value:     {p_value:.4f}")

Z-statistic: 3.1644
P-value:     0.0016


In [34]:
from statsmodels.stats.proportion import proportion_confint

p30 = r7_30 / n30
p40 = r7_40 / n40
diff = p30 - p40

# 95% CI on the difference (via normal approximation)
se_diff = np.sqrt(p30*(1-p30)/n30 + p40*(1-p40)/n40)
ci_low  = diff - 1.96 * se_diff
ci_high = diff + 1.96 * se_diff

print(f"Observed difference: {diff:.4f} ({diff*100:.2f} pp)")
print(f"95% CI: [{ci_low:.4f}, {ci_high:.4f}]")


Observed difference: 0.0082 (0.82 pp)
95% CI: [0.0031, 0.0133]


## Observations

* p = 0.0016 < 0.05, so we reject $H_0$: there is statistically significant evidence that gate position affects Day-7 retention.
* The 95% confidence interval for the difference excludes 0, confirming the effect is unlikely to be due to chance.
* For every 10,000 players, gate_30 retains ~82 more users after 7 days than gate_40.

# Practical significance

Statistical significance only tells us the difference is real, not whether it matters for the business. Here we size the effect in absolute and relative terms.

In [35]:
# Day-7 (primary metric)
abs_lift  = p30 - p40                    # 0.0082
rel_lift  = (p30 - p40) / p40 * 100   # relative % improvement

print("absolute lift: ", abs_lift)
print("Relative lift: ", rel_lift)

absolute lift:  0.008201298315205913
Relative lift:  4.506206776910275


# Segmentation Analysis

The global effect could be an average masking very different behavior across player types. Here we re-run the Day-7 retention test within each engagement-depth bucket (gamerounds played) to see where the effect actually comes from, applying a Bonferroni correction since we are now running six tests instead of one.

In [36]:
from statsmodels.stats.multitest import multipletests

segments = df["grouped_gamerounds"].cat.categories.tolist()
results = []

for seg in segments:
    mask = df["grouped_gamerounds"] == seg 
    g30 = df[mask & (df["version"] == "gate_30")]
    g40 = df[mask & (df["version"] == "gate_40")]
    n30, n40 = len(g30), len(g40)
    x30, x40 = g30["retention_7"].sum(), g40["retention_7"].sum()
    p30, p40 = x30/n30, x40/n40
    diff = p30 - p40
    z, p_raw   = proportions_ztest([x30, x40], [n30, n40], alternative="two-sided")
    results.append({
        "segment": seg,
        "n_gate30": n30, "n_gate40": n40,
        "p30": round(p30, 4), "p40": round(p40, 4),
        "diff_pp": round(diff * 100, 3),
        "z_stat": round(z, 3),
        "p_raw": round(p_raw, 4),
    })
results_df = pd.DataFrame(results)
    


In [37]:
results_df.head()

,segment,n_gate30,n_gate40,p30,p40,diff_pp,z_stat,p_raw
0,0-5,10119,10604,0.0129,0.0144,-0.148,-0.918,0.3588
1,6-20,12428,12657,0.0503,0.0475,0.281,1.031,0.3027
2,21-50,9080,8495,0.1607,0.1456,1.507,2.770,0.0056
3,51-100,5041,5386,0.3809,0.3331,4.779,5.093,0.0000
4,101-500,5668,5861,0.6962,0.6941,0.211,0.246,0.8057


In [38]:
# apply Bonferroni correction
reject, p_corrected, _, _ = multipletests(
    results_df["p_raw"], alpha=0.05, method="bonferroni"
)

results_df["p_corrected"] = p_corrected.round(4)
results_df["reject_H0"]   = reject

print(results_df[[
    "segment", "n_gate30", "n_gate40",
    "p30", "p40", "diff_pp",
    "p_raw", "p_corrected", "reject_H0"
]])

   segment  n_gate30  n_gate40     p30     p40  diff_pp   p_raw  p_corrected  \
0      0-5     10119     10604  0.0129  0.0144   -0.148  0.3588       1.0000   
1     6-20     12428     12657  0.0503  0.0475    0.281  0.3027       1.0000   
2    21-50      9080      8495  0.1607  0.1456    1.507  0.0056       0.0336   
3   51-100      5041      5386  0.3809  0.3331    4.779  0.0000       0.0000   
4  101-500      5668      5861  0.6962  0.6941    0.211  0.8057       1.0000   
5     500+       427       429  0.9485  0.9627   -1.423  0.3123       1.0000   

   reject_H0  
0      False  
1      False  
2       True  
3       True  
4      False  
5      False  


In [ ]:
# visualize the effect

import matplotlib.patches as mpatches

SIG_COLOR = "#2a78d6"      # significant after Bonferroni correction
NOTSIG_COLOR = "#898781"   # not significant — muted, not "bad"

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

colors = [SIG_COLOR if r else NOTSIG_COLOR for r in results_df["reject_H0"]]

# Left: retention rate per segment per group
x = np.arange(len(segments))
w = 0.35
axs[0].bar(x - w/2, results_df["p30"], w, label=GROUP_LABELS["gate_30"], color=GROUP_COLORS["gate_30"], alpha=0.9)
axs[0].bar(x + w/2, results_df["p40"], w, label=GROUP_LABELS["gate_40"], color=GROUP_COLORS["gate_40"], alpha=0.9)
axs[0].set_xticks(x)
axs[0].set_xticklabels(segments, rotation=25)
axs[0].set_xlabel("Gamerounds bucket")
axs[0].set_ylabel("Day-7 retention rate")
axs[0].set_title("Retention rate by segment")
axs[0].legend(title="Group")

# Right: difference (gate_30 - gate_40) per segment, colored by significance
bars = axs[1].bar(x, results_df["diff_pp"], color=colors, alpha=0.9)
axs[1].axhline(0, color="black", lw=1, linestyle="--")
axs[1].set_xticks(x)
axs[1].set_xticklabels(segments, rotation=25)
axs[1].set_xlabel("Gamerounds bucket")
axs[1].set_ylabel("Difference in Day-7 retention (pp)\ngate_30 − gate_40")
axs[1].set_title("Where the effect comes from, by segment")
axs[1].legend(handles=[
    mpatches.Patch(color=SIG_COLOR, label="Significant (Bonferroni-adjusted p < 0.05)"),
    mpatches.Patch(color=NOTSIG_COLOR, label="Not significant"),
], loc="upper right", fontsize=9)

for bar, row in zip(bars, results_df.itertuples()):
    offset = 0.15 if bar.get_height() >= 0 else -0.35
    axs[1].text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + offset,
                f"{row.diff_pp:+.2f} pp\np={row.p_corrected:.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.show()

## Observations

* The global 0.82 pp advantage for gate_30 is not spread evenly — it's driven almost entirely by the 21-100 round segment, which represents players on the retention fence.
* Players with very low (0-20) or very high (100+) engagement show no significant difference between groups: they either churn before reaching the gate or are already committed to the game regardless of it.

# Decision and recommendations

In [ ]:
# --- key numbers from previous phases ---
p30_7, p40_7   = 0.1902, 0.1820
abs_lift_7     = p30_7 - p40_7           # 0.0082
rel_lift_7     = abs_lift_7 / p40_7      # ~4.5%

print("=== Evidence Summary ===")
print(f"Day-7 absolute lift : {abs_lift_7*100:.2f} pp  (gate_30 favored)")
print(f"Day-7 relative lift : {rel_lift_7*100:.2f}%")
print(f"95% CI on diff      : [0.0031, 0.0133]  — excludes 0")
print(f"Significant segments: 21-50 rounds (1.51 pp), 51-100 rounds (4.78 pp)")
print(f"Unaffected segments : 0-20 rounds (pre-gate churn), 100+ rounds (committed players)")

In [41]:
monthly_new_players = 1_000_000          # hypothetical 

retained_extra_per_10k = abs_lift_7 * 10_000
retained_extra_monthly = abs_lift_7 * monthly_new_players

print("=== Business Impact ===")
print(f"Extra Day-7 retained users per 10k : {retained_extra_per_10k:.0f}")
print(f"Extra Day-7 retained users / month : {retained_extra_monthly:,.0f}")


=== Business Impact ===
Extra Day-7 retained users per 10k : 82
Extra Day-7 retained users / month : 8,200


## Decision

**Recommendation: retain gate_30 (do not move the gate to level 40).**

### Evidence
- Gate_30 produces a statistically significant improvement in Day-7 retention
  (z = 3.16, p = 0.0016, 95% CI [0.31 pp, 1.33 pp]).
- The relative lift of ~4.5% on a base rate of 18.2% is practically meaningful
  for a mobile game where retention is difficult to move.
- The effect is concentrated in 21-100 round players — the segment that
  actually reaches the gate and is behaviorally on the fence. This gives
  the result a credible mechanism: earlier forced breaks reduce fatigue
  before the 7-day mark.
- High-engagement players (100+ rounds) are unaffected, so moving the gate
  carries no upside for that segment.

### Limitations
- Day-7 retention is a proxy. Ideally this would be validated against
  monetization or Day-30 retention data.
- The 500+ segment showed a non-significant reversal (gate_40 slightly better).
  Too small a sample to act on, but worth monitoring.
- This dataset does not allow causal claims about *why* gate_30 works —
  only that it does.

### Follow-up
- Run a targeted experiment on the 100+ round segment with a gate at level 35
  or 40 to test whether a later gate could improve retention for power users
  without harming mid-engagement players.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5))

# 1) Global Day-7 retention by group
groups = ["gate_30", "gate_40"]
ret_day7 = [p30_7, p40_7]
bars0 = axs[0].bar(groups, ret_day7, color=[GROUP_COLORS[g] for g in groups], alpha=0.9)
axs[0].set_xticks(range(len(groups)))
axs[0].set_xticklabels([GROUP_LABELS[g] for g in groups])
axs[0].set_ylabel("Day-7 retention rate")
axs[0].set_title("Global retention by group")
for bar, val in zip(bars0, ret_day7):
    axs[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", fontsize=11)

# 2) Segment effect (diff_pp), reusing the significance results computed above
colors = [SIG_COLOR if r else NOTSIG_COLOR for r in results_df["reject_H0"]]
axs[1].bar(results_df["segment"], results_df["diff_pp"], color=colors, alpha=0.9)
axs[1].axhline(0, color="black", lw=1, linestyle="--")
axs[1].set_xlabel("Gamerounds bucket")
axs[1].set_ylabel("Retention diff (pp)\ngate_30 − gate_40")
axs[1].set_title("Effect by engagement segment")
axs[1].tick_params(axis="x", rotation=25)
axs[1].legend(handles=[
    mpatches.Patch(color=SIG_COLOR, label="Significant"),
    mpatches.Patch(color=NOTSIG_COLOR, label="Not significant"),
], fontsize=9)

# 3) Overall effect size with 95% CI
axs[2].barh(["Day-7 retention"], [abs_lift_7 * 100], color=SIG_COLOR, alpha=0.9)
axs[2].errorbar([abs_lift_7 * 100], ["Day-7 retention"],
                xerr=[[abs_lift_7*100 - ci_low*100], [ci_high*100 - abs_lift_7*100]],
                fmt="none", color="black", capsize=6, lw=2)
axs[2].axvline(0, color=NOTSIG_COLOR, lw=1.5, linestyle="--")
axs[2].set_xlabel("Absolute lift, gate_30 − gate_40 (pp)")
axs[2].set_title("Overall effect size with 95% CI")

plt.suptitle("A/B test summary — Cookie Cats gate position", fontsize=14, y=1.04)
plt.tight_layout()
plt.show()